In [1]:
import torch
from torch import nn


In [2]:
class TinyNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.w1 = nn.Parameter(torch.tensor([[0.5]]))
        self.b1 = nn.Parameter(torch.tensor([0.0]))

        self.w2 = nn.Parameter(torch.tensor([[-0.5]]))
        self.b2 = nn.Parameter(torch.tensor([0.0]))

        self.v1 = nn.Parameter(torch.tensor([[1.0]]))
        self.v2 = nn.Parameter(torch.tensor([[1.0]]))

        self.b3 = nn.Parameter(torch.tensor([0.0]))

    def forward(self, x):

        z1 = x * self.w1 + self.b1
        a1 = torch.tanh(z1)

        z2 = x * self.w2 + self.b2
        a2 = torch.tanh(z2)

        y_hat = a1 * self.v1 + a2 * self.v2 + self.b3

        return y_hat


In [3]:
# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = TinyNet()

x = torch.tensor([[1.0]])
y = torch.tensor([[1.0]])


In [4]:
# ------------------------------------------------------------
# Create vanilla gradient descent optimizer
# ------------------------------------------------------------

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1
)

In [5]:
# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

y_hat = model(x)

loss = (y_hat - y).pow(2)

print("Before optimizer.step()")
print("----------------------")
print("prediction:", y_hat.item())
print("loss:", loss.item())


Before optimizer.step()
----------------------
prediction: 0.0
loss: 1.0


In [6]:
# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

loss.backward()


In [7]:
# ------------------------------------------------------------
# Inspect gradients
# ------------------------------------------------------------

print()
print("Gradients")
print("----------------------")

for name, parameter in model.named_parameters():
    print(f"{name:3s}: {parameter.grad.item(): .6f}")



Gradients
----------------------
w1 : -1.572895
b1 : -1.572895
w2 : -1.572895
b2 : -1.572895
v1 : -0.924234
v2 :  0.924234
b3 : -2.000000


In [8]:
# ------------------------------------------------------------
# Save parameters BEFORE optimizer.step()
# ------------------------------------------------------------

old_parameters = {}

for name, parameter in model.named_parameters():
    old_parameters[name] = parameter.detach().clone()



In [9]:
# ------------------------------------------------------------
# Optimizer step
# ------------------------------------------------------------

optimizer.step()


In [10]:
# ------------------------------------------------------------
# Compare old vs new
# ------------------------------------------------------------

print()
print("Parameter updates")
print("----------------------")

for name, parameter in model.named_parameters():

    old_value = old_parameters[name].item()
    gradient = parameter.grad.item()
    new_value = parameter.item()

    manual_value = old_value - 0.1 * gradient

    print(f"{name}:")
    print(f"  old:    {old_value: .6f}")
    print(f"  grad:   {gradient: .6f}")
    print(f"  manual: {manual_value: .6f}")
    print(f"  PyTorch:{new_value: .6f}")
    print()



Parameter updates
----------------------
w1:
  old:     0.500000
  grad:   -1.572895
  manual:  0.657290
  PyTorch: 0.657290

b1:
  old:     0.000000
  grad:   -1.572895
  manual:  0.157290
  PyTorch: 0.157290

w2:
  old:    -0.500000
  grad:   -1.572895
  manual: -0.342710
  PyTorch:-0.342710

b2:
  old:     0.000000
  grad:   -1.572895
  manual:  0.157290
  PyTorch: 0.157290

v1:
  old:     1.000000
  grad:   -0.924234
  manual:  1.092423
  PyTorch: 1.092423

v2:
  old:     1.000000
  grad:    0.924234
  manual:  0.907577
  PyTorch: 0.907577

b3:
  old:     0.000000
  grad:   -2.000000
  manual:  0.200000
  PyTorch: 0.200000

